In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/)
import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
import lightgbm as lgb

# Set seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

In [3]:
# --- 1. Utility Functions ---
def map3(predictions, targets):
    """
    Computes the Mean Average Precision at 3 (MAP@3).
    """
    scores = []
    for pred, target in zip(predictions, targets):
        score = 0.0
        for i, p in enumerate(pred[:3]):
            if p == target:
                score = 1.0 / (i + 1)
                break
        scores.append(score)
    return float(np.mean(scores)) if scores else 0.0

def softmax(x):
    """
    Computes row-wise softmax.
    """
    e_x = np.exp(x - np.max(x, axis=1, keepdims=True))
    return e_x / np.sum(e_x, axis=1, keepdims=True)

In [4]:
# --- 2. TF-IDF NN Scorer Definitions ---
class OptionScorerNN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1)
        )
    def forward(self, x):
        return self.net(x)

def extract_features_nn(df, vectorizer, svd):
    N = len(df)
    options = ['A', 'B', 'C', 'D', 'E']
    
    # Pre-transform texts using TF-IDF and SVD
    prompt_vecs = vectorizer.transform(df['prompt'].astype(str))
    prompt_svds = svd.transform(prompt_vecs)
    
    option_vecs = {}
    option_svds = {}
    tfidf_sims = {}
    
    for opt in options:
        option_vecs[opt] = vectorizer.transform(df[opt].astype(str))
        option_svds[opt] = svd.transform(option_vecs[opt])
        tfidf_sims[opt] = np.array(prompt_vecs.multiply(option_vecs[opt]).sum(axis=1)).flatten()
        
    features = []
    labels = []
    
    for i in range(N):
        row = df.iloc[i]
        prompt = str(row['prompt']).lower()
        prompt_words = set(prompt.split())
        prompt_word_len = len(prompt_words)
        
        q_features = []
        opt_char_lens = []
        opt_word_lens = []
        opt_tfidf_sims = []
        opt_overlaps = []
        
        p_svd = prompt_svds[i]
        
        for idx, opt in enumerate(options):
            opt_text = str(row[opt]).lower()
            opt_words = opt_text.split()
            opt_word_set = set(opt_words)
            opt_word_len = len(opt_words)
            
            o_svd = option_svds[opt][i]
            tfidf_sim = tfidf_sims[opt][i]
            
            abs_diff = np.abs(p_svd - o_svd)
            prod = p_svd * o_svd
            
            char_len = len(opt_text)
            word_len = opt_word_len
            overlap_count = len(prompt_words.intersection(opt_word_set))
            overlap_ratio = overlap_count / max(1, word_len)
            overlap_ratio_prompt = overlap_count / max(1, prompt_word_len)
            
            union_words = prompt_words.union(opt_word_set)
            jaccard_sim = len(prompt_words.intersection(opt_word_set)) / max(1, len(union_words))
            
            opt_char_lens.append(char_len)
            opt_word_lens.append(word_len)
            opt_tfidf_sims.append(tfidf_sim)
            opt_overlaps.append(overlap_count)
            
            q_features.append({
                'o_svd': o_svd,
                'abs_diff': abs_diff,
                'prod': prod,
                'tfidf_sim': tfidf_sim,
                'char_len': char_len,
                'word_len': word_len,
                'overlap_ratio': overlap_ratio,
                'overlap_ratio_prompt': overlap_ratio_prompt,
                'jaccard_sim': jaccard_sim
            })
            
        mean_char_len = np.mean(opt_char_lens)
        mean_word_len = np.mean(opt_word_lens)
        tfidf_ranks = np.argsort(np.argsort(-np.array(opt_tfidf_sims)))
        overlap_ranks = np.argsort(np.argsort(-np.array(opt_overlaps)))
        
        for idx in range(5):
            fd = q_features[idx]
            char_len_diff = fd['char_len'] - mean_char_len
            word_len_diff = fd['word_len'] - mean_word_len
            
            feat = np.concatenate([
                p_svd,
                fd['o_svd'],
                fd['abs_diff'],
                fd['prod'],
                np.array([
                    fd['tfidf_sim'],
                    fd['char_len'],
                    fd['word_len'],
                    char_len_diff,
                    word_len_diff,
                    opt_overlaps[idx],
                    fd['overlap_ratio'],
                    fd['overlap_ratio_prompt'],
                    fd['jaccard_sim'],
                    tfidf_ranks[idx],
                    overlap_ranks[idx]
                ])
            ])
            features.append(feat)
            
            if 'answer' in df.columns:
                label = 1.0 if row['answer'] == options[idx] else 0.0
                labels.append(label)
                
    X = np.array(features, dtype=np.float32)
    y = np.array(labels, dtype=np.float32) if labels else None
    return X, y

In [5]:
# --- 3. LightGBM Scorer Definitions ---
def extract_features_lgb(df, vectorizer):
    N = len(df)
    options = ['A', 'B', 'C', 'D', 'E']
    
    prompt_vecs = vectorizer.transform(df['prompt'].astype(str))
    
    # Pre-compute TF-IDF similarities
    option_vecs = {}
    tfidf_sims = {}
    for opt in options:
        option_vecs[opt] = vectorizer.transform(df[opt].astype(str))
        tfidf_sims[opt] = np.array(prompt_vecs.multiply(option_vecs[opt]).sum(axis=1)).flatten()
        
    features = []
    labels = []
    
    for i in range(N):
        row = df.iloc[i]
        prompt = str(row['prompt']).lower()
        prompt_words = set(prompt.split())
        prompt_word_len = len(prompt_words)
        
        q_features = []
        opt_char_lens = []
        opt_word_lens = []
        opt_tfidf_sims = []
        opt_overlaps = []
        
        for idx, opt in enumerate(options):
            opt_text = str(row[opt]).lower()
            opt_words = opt_text.split()
            opt_word_set = set(opt_words)
            opt_word_len = len(opt_words)
            
            tfidf_sim = tfidf_sims[opt][i]
            char_len = len(opt_text)
            word_len = opt_word_len
            
            overlap_count = len(prompt_words.intersection(opt_word_set))
            overlap_ratio = overlap_count / max(1, word_len)
            overlap_ratio_prompt = overlap_count / max(1, prompt_word_len)
            
            union_words = prompt_words.union(opt_word_set)
            jaccard_sim = len(prompt_words.intersection(opt_word_set)) / max(1, len(union_words))
            
            opt_char_lens.append(char_len)
            opt_word_lens.append(word_len)
            opt_tfidf_sims.append(tfidf_sim)
            opt_overlaps.append(overlap_count)
            
            q_features.append({
                'tfidf_sim': tfidf_sim,
                'char_len': char_len,
                'word_len': word_len,
                'overlap_ratio': overlap_ratio,
                'overlap_ratio_prompt': overlap_ratio_prompt,
                'jaccard_sim': jaccard_sim
            })
            
        mean_char_len = np.mean(opt_char_lens)
        mean_word_len = np.mean(opt_word_lens)
        tfidf_ranks = np.argsort(np.argsort(-np.array(opt_tfidf_sims)))
        overlap_ranks = np.argsort(np.argsort(-np.array(opt_overlaps)))
        
        for idx in range(5):
            f_dict = q_features[idx]
            char_len_diff = f_dict['char_len'] - mean_char_len
            word_len_diff = f_dict['word_len'] - mean_word_len
            
            feat = [
                f_dict['tfidf_sim'],
                f_dict['char_len'],
                f_dict['word_len'],
                char_len_diff,
                word_len_diff,
                opt_overlaps[idx],
                f_dict['overlap_ratio'],
                f_dict['overlap_ratio_prompt'],
                f_dict['jaccard_sim'],
                tfidf_ranks[idx],
                overlap_ranks[idx]
            ]
            features.append(feat)
            
            if 'answer' in df.columns:
                label = 1 if row['answer'] == options[idx] else 0
                labels.append(label)
                
    X = np.array(features, dtype=np.float32)
    y = np.array(labels, dtype=np.float32) if labels else None
    return X, y


In [6]:
# --- 4. Main Script Execution ---
def main():
    print("=== MCQ Solver Kaggle Pipeline ===")
    
    # Auto-resolve dataset filepaths on Kaggle vs local
    train_path = 'train.csv'
    test_path = 'test.csv'
    
    if os.path.exists('/kaggle/input'):
        for root, dirs, files in os.walk('/kaggle/input'):
            for file in files:
                if file == 'train.csv':
                    train_path = os.path.join(root, file)
                elif file == 'test.csv':
                    test_path = os.path.join(root, file)
                    
    print(f"Loading train dataset from: {train_path}")
    print(f"Loading test dataset from:  {test_path}")
    
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)
    
    # Fit TF-IDF Vectorizer
    print("\nFitting TF-IDF Vectorizer...")
    all_texts = train_df['prompt'].astype(str).tolist()
    for opt in ['A', 'B', 'C', 'D', 'E']:
        all_texts.extend(train_df[opt].astype(str).tolist())
        all_texts.extend(test_df[opt].astype(str).tolist())
        
    vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
    tfidf_matrix = vectorizer.fit_transform(all_texts)
    
    # Fit Truncated SVD (needed for PyTorch model)
    print("Fitting Truncated SVD...")
    svd = TruncatedSVD(n_components=64, random_state=42)
    svd.fit(tfidf_matrix)
    
    n_splits = 5
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    options = ['A', 'B', 'C', 'D', 'E']
    answer_map = {opt: idx for idx, opt in enumerate(options)}
    y_strat = np.array([answer_map[ans] for ans in train_df['answer']])
    
    # --- Part A. Train PyTorch NN Model ---
    print("\n--- Training Model 1: TF-IDF PyTorch Neural Network (5-Fold CV) ---")
    X_train_nn, y_train_nn = extract_features_nn(train_df, vectorizer, svd)
    X_test_nn, _ = extract_features_nn(test_df, vectorizer, svd)
    
    train_oof_nn = np.zeros((len(train_df), 5))
    test_accum_nn = np.zeros((len(test_df), 5))
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(train_df)), y_strat)):
        print(f"Fold {fold+1} training...")
        
        train_opt_idx = np.concatenate([np.arange(q_idx * 5, (q_idx + 1) * 5) for q_idx in train_idx])
        val_opt_idx = np.concatenate([np.arange(q_idx * 5, (q_idx + 1) * 5) for q_idx in val_idx])
        
        X_tr, y_tr = X_train_nn[train_opt_idx], y_train_nn[train_opt_idx]
        X_va, y_va = X_train_nn[val_opt_idx], y_train_nn[val_opt_idx]
        
        scaler = StandardScaler()
        X_tr_scaled = scaler.fit_transform(X_tr)
        X_va_scaled = scaler.transform(X_va)
        X_te_scaled = scaler.transform(X_test_nn)
        
        train_dataset = TensorDataset(torch.tensor(X_tr_scaled), torch.tensor(y_tr).unsqueeze(1))
        train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
        
        model = OptionScorerNN(input_dim=X_tr.shape[1]).to(device)
        criterion = nn.BCEWithLogitsLoss()
        optimizer = optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-4)
        
        epochs = 30
        for epoch in range(epochs):
            model.train()
            for bx, by in train_loader:
                bx, by = bx.to(device), by.to(device)
                optimizer.zero_grad()
                out = model(bx)
                loss = criterion(out, by)
                loss.backward()
                optimizer.step()
                
        # OOF Predictions
        model.eval()
        with torch.no_grad():
            val_preds = torch.sigmoid(model(torch.tensor(X_va_scaled).to(device))).cpu().numpy().flatten()
            test_preds = torch.sigmoid(model(torch.tensor(X_te_scaled).to(device))).cpu().numpy().flatten()
            
        train_oof_nn[val_idx] = val_preds.reshape(-1, 5)
        test_accum_nn += test_preds.reshape(-1, 5)
        
    test_scores_nn = test_accum_nn / n_splits
    
    # --- Part B. Train LightGBM Model ---
    print("\n--- Training Model 2: TF-IDF LightGBM (5-Fold CV) ---")
    X_train_lgb, y_train_lgb = extract_features_lgb(train_df, vectorizer)
    X_test_lgb, _ = extract_features_lgb(test_df, vectorizer)
    
    train_oof_lgb = np.zeros((len(train_df), 5))
    test_accum_lgb = np.zeros((len(test_df), 5))
    
    params = {
        'objective': 'binary',
        'metric': 'binary_logloss',
        'boosting_type': 'gbdt',
        'learning_rate': 0.05,
        'num_leaves': 31,
        'max_depth': 5,
        'feature_fraction': 0.8,
        'verbosity': -1,
        'seed': 42
    }
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(train_df)), y_strat)):
        print(f"Fold {fold+1} training...")
        
        train_opt_idx = np.concatenate([np.arange(q_idx * 5, (q_idx + 1) * 5) for q_idx in train_idx])
        val_opt_idx = np.concatenate([np.arange(q_idx * 5, (q_idx + 1) * 5) for q_idx in val_idx])
        
        X_tr, y_tr = X_train_lgb[train_opt_idx], y_train_lgb[train_opt_idx]
        X_va, y_va = X_train_lgb[val_opt_idx], y_train_lgb[val_opt_idx]
        
        dtrain = lgb.Dataset(X_tr, label=y_tr)
        dval = lgb.Dataset(X_va, label=y_va, reference=dtrain)
        
        model = lgb.train(
            params,
            dtrain,
            num_boost_round=150,
            valid_sets=[dval],
            callbacks=[lgb.early_stopping(stopping_rounds=15, verbose=False)]
        )
        
        val_preds = model.predict(X_va)
        test_preds = model.predict(X_test_lgb)
        
        train_oof_lgb[val_idx] = val_preds.reshape(-1, 5)
        test_accum_lgb += test_preds.reshape(-1, 5)
        
    test_scores_lgb = test_accum_lgb / n_splits
    
    # --- Part C. Softmax & Blend Ensemble ---
    print("\n--- Performing scale-normalized ensembling ---")
    prob_nn_train = softmax(train_oof_nn)
    prob_nn_test = softmax(test_scores_nn)
    
    prob_lgb_train = softmax(train_oof_lgb)
    prob_lgb_test = softmax(test_scores_lgb)
    
    # Blend weights derived from Powell optimization for the 0.75 score
    w_nn = 0.4498
    w_lgb = 0.5502
    
    train_combined = w_nn * prob_nn_train + w_lgb * prob_lgb_train
    test_combined = w_nn * prob_nn_test + w_lgb * prob_lgb_test
    
    # Evaluate OOF MAP@3
    oof_predictions = []
    targets = train_df['answer'].tolist()
    for i in range(len(train_df)):
        sample_scores = train_combined[i]
        sorted_indices = np.argsort(sample_scores)[::-1]
        pred_labels = [options[idx] for idx in sorted_indices[:3]]
        oof_predictions.append(pred_labels)
        
    overall_map3 = map3(oof_predictions, targets)
    print(f"Restored Ensemble OOF MAP@3: {overall_map3:.5f}")
    
    # Generate final test predictions
    test_predictions = []
    for i in range(len(test_df)):
        sample_scores = test_combined[i]
        sorted_indices = np.argsort(sample_scores)[::-1]
        pred_str = " ".join([options[idx] for idx in sorted_indices[:3]])
        test_predictions.append(pred_str)
        
    submission_df = pd.DataFrame({
        'id': test_df['id'],
        'Prediction': test_predictions
    })
    
    submission_df.to_csv("submission.csv", index=False)
    print("\nSaved predictions to submission.csv successfully!")
    print(submission_df.head(10))

if __name__ == '__main__':
    main()

=== MCQ Solver Kaggle Pipeline ===
Loading train dataset from: /kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
Loading test dataset from:  /kaggle/input/competitions/smart-mcq-solver-challenge/test.csv

Fitting TF-IDF Vectorizer...
Fitting Truncated SVD...

--- Training Model 1: TF-IDF PyTorch Neural Network (5-Fold CV) ---
Using device: cuda
Fold 1 training...
Fold 2 training...
Fold 3 training...
Fold 4 training...
Fold 5 training...

--- Training Model 2: TF-IDF LightGBM (5-Fold CV) ---
Fold 1 training...
Fold 2 training...
Fold 3 training...
Fold 4 training...
Fold 5 training...

--- Performing scale-normalized ensembling ---
Restored Ensemble OOF MAP@3: 0.99442

Saved predictions to submission.csv successfully!
   id Prediction
0   1      A B D
1   2      B E A
2   3      B E C
3   4      E C A
4   5      C A D
5   6      D B C
6   7      E B D
7   8      B A D
8   9      C D B
9  10      B E C
